# Financial Transaction Risk & Anomaly Engine - Exploratory Data Analysis

This notebook contains the exploratory data analysis (EDA) and data quality assessment for the Financial Transaction Risk & Anomaly Engine. The objective is to understand features, inspect data health, and detect patterns that differentiate normal transactions from anomalies.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure package resolution relative to parent directory
sys.path.append(os.path.abspath('..'))
from src import config
from src.utils import load_dataset

## 1. Loading the Dataset
We load the cleaned dataset generated in the preprocessing step (`data/processed/transactions_clean.csv`).

In [ ]:
df = load_dataset(config.PROCESSED_DATA_PATH)
df.head()

## 2. Feature Characterization
We analyze the data features and classify them as identifiers, numerical features, categorical features, or the target variable.

In [ ]:
print("=== Dataset Dimensions ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\n=== Feature Classifications ===")
print("Identifiers:  ['transaction_id', 'customer_id', 'timestamp']")
print("Numerical:    ['amount']")
print("Categorical:  ['merchant_category', 'location', 'device_type']")
print("Target:       'is_anomaly'")
print("\n=== Data Types ===")
print(df.dtypes)

## 3. Data Quality Report
We inspect the dataset for null values, unique counts, duplicates, constant columns, and check the cardinality of categorical fields.

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Unique Values ===")
print(df.nunique())

print(f"\n=== Duplicate Rows: {df.duplicated().sum()} ===")

# Check for constant columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"\n=== Constant Columns: {constant_cols} ===")

# Check cardinality for categorical fields
categorical = ['merchant_category', 'location', 'device_type']
print("\n=== Categorical Cardinality ===")
for col in categorical:
    print(f"  {col}: {df[col].nunique()} unique values")

### Data Quality Observations
- **No Missing Values**: The dataset is completely populated, meaning no imputation is required.
- **No Duplicate Rows**: All transactions represent unique occurrences.
- **Low Cardinality**: None of our categorical features display high cardinality, which simplifies encoding schemas (e.g. One-Hot encoding can be safely utilized without feature explosion).

## 4. Visualizations

### 4.1. Class Distribution (Target Imbalance)

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='is_anomaly', hue='is_anomaly', data=df, palette={0: "#4F46E5", 1: "#EF4444"}, legend=False)
plt.title("Transaction Class Distribution (Imbalance Check)", pad=15)
plt.xlabel("Is Anomaly")
plt.ylabel("Count")

total = len(df)
for p in ax.patches:
    height = p.get_height()
    if pd.isna(height) or height == 0: continue
    percentage = 100 * height / total
    ax.annotate(f'{int(height)}\n({percentage:.2f}%)', 
                (p.get_x() + p.get_width() / 2., height - (height * 0.2 if height > 1000 else -20)),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points',
                color='white' if height > 1000 else 'black', fontweight='bold')
plt.show()

**Observation**: The target classes are heavily imbalanced. Anomalous transactions represent only **1.50%** of the dataset (150 occurrences), while **98.50%** are normal (9,850 occurrences). Our modeling pipeline must address this using appropriate weighting, resampling, or ranking metrics (e.g. F1-score, Precision-Recall AUC) rather than standard accuracy.

### 4.2. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall Log Amount
df_log = df.copy()
df_log['log_amount'] = np.log10(df_log['amount'] + 1)
sns.histplot(x='log_amount', data=df_log, kde=True, color="#4F46E5", ax=axes[0])
axes[0].set_title("Overall Log-Amount Distribution")
axes[0].set_xlabel("Log10(Amount + 1)")

# Amount by Target Class
sns.boxplot(x='is_anomaly', y='amount', hue='is_anomaly', data=df, palette={0: "#4F46E5", 1: "#EF4444"}, ax=axes[1], legend=False)
axes[1].set_title("Transaction Amount by Anomaly Class")
axes[1].set_xlabel("Is Anomaly")
axes[1].set_ylabel("Amount ($)")
axes[1].set_yscale('log')
plt.show()

**Observation**: The distribution of normal transaction amounts spans a typical everyday retail envelope ($10 - $500). Anomalous transactions are skewed towards significantly higher amounts, with distinct high-value patterns reaching up to $20,000, as shown by the log-scaled boxplot.

### 4.3. Top Transaction Categories

In [ ]:
plt.figure(figsize=(9, 5))
order = df['merchant_category'].value_counts().index
colors = ["#4F46E5" if c not in ["transfer", "cash_withdrawal", "travel"] else "#818CF8" for c in order]
sns.countplot(y='merchant_category', hue='merchant_category', data=df, order=order, palette=colors, legend=False)
plt.title("Transaction Frequency by Merchant Category", pad=15)
plt.xlabel("Count")
plt.ylabel("Merchant Category")
plt.show()

**Observation**: General dining and grocery transactions represent the bulk of normal client activity. Transfers, cash withdrawals, and travel are less frequent but critical for financial risk profiling.

### 4.4. Feature Correlation Heatmap

In [ ]:
df_encoded = df.copy()
df_corr_input = pd.get_dummies(df_encoded.drop(columns=['transaction_id', 'customer_id', 'timestamp']), columns=categorical, drop_first=False)
for col in df_corr_input.select_dtypes(include=['bool']).columns:
    df_corr_input[col] = df_corr_input[col].astype(int)

corr_matrix = df_corr_input.corr()
top_features = ['amount', 'is_anomaly'] + [c for c in corr_matrix.columns if 'merchant_category' in c or 'device_type' in c]
subset_corr = corr_matrix.loc[top_features, top_features]

plt.figure(figsize=(10, 8))
sns.heatmap(subset_corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1.0, vmax=1.0)
plt.title("Correlation Matrix of Features")
plt.show()

**Observation**: The transaction `amount` displays a moderate positive linear correlation with `is_anomaly` (~0.33), supporting that financial scale is a key indicator of anomaly status. Cash withdrawals and web/mobile channels show slightly higher correlation associations with target risk compared to terminal card-present swipes.

### 4.5. Device Type and Location Anomaly Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Device Anomaly Rates
device_anomaly = df.groupby('device_type')['is_anomaly'].mean().reset_index()
device_anomaly['is_anomaly'] = device_anomaly['is_anomaly'] * 100
sns.barplot(x='device_type', y='is_anomaly', hue='device_type', data=device_anomaly, palette="Purples_r", ax=axes[0], legend=False)
axes[0].set_title("Anomaly Rate (%) by Device Type")
axes[0].set_ylabel("Anomaly Rate (%)")

# Location Anomaly Rates
top_locs = df['location'].value_counts().index[:10]
df_top_loc = df[df['location'].isin(top_locs)]
loc_anomaly = df_top_loc.groupby('location')['is_anomaly'].mean().reset_index()
loc_anomaly['is_anomaly'] = loc_anomaly['is_anomaly'] * 100
loc_anomaly = loc_anomaly.sort_values(by='is_anomaly', ascending=False)
sns.barplot(x='is_anomaly', y='location', hue='location', data=loc_anomaly, palette="Reds_r", ax=axes[1], legend=False)
axes[1].set_title("Anomaly Rate (%) by Top 10 Locations")
axes[1].set_xlabel("Anomaly Rate (%)")
plt.tight_layout()
plt.show()

**Observation**: Anomaly rates differ heavily across dimensions. Web and mobile transactions have significantly higher anomaly rates than Point-Of-Sale (POS) terminal transactions. In addition, international card transitions (London, Paris, Tokyo, etc.) show extremely high anomaly rates compared to domestic locations.

## 5. Business Insights

Based on the exploratory data analysis, we establish the following insights regarding transaction risks and anomaly characteristics:

1. **Severe Target Imbalance (1.50% Risk Baseline)**: Out of 10,000 transactions, only 150 are classified as anomalies. This highlights that normal user behavior dominates the stream. The model must focus heavily on precision/recall dynamics to minimize both false negatives (untracked risk) and false positives (client friction).
2. **Critical Value Threshold (High-Amount Flags)**: While normal transactions average between $30 and $200, anomalous transactions show a huge concentration in values above $1,500, stretching up to $20,000. Implementing an immediate risk score multiplier for transactions exceeding $1,000 is strongly recommended.
3. **Geographical Cross-Border Risks**: Domestic transactions (e.g. New York, Los Angeles) carry low anomaly rates (around 1%). Conversely, international transactions (e.g., Tokyo, Mumbai, London, Paris) exhibit anomaly rates above 5%, making cross-border geolocation changes a high-priority risk feature.
4. **Web and Mobile Channel Vulnerability**: Device type is highly descriptive of risk. Web transactions display the highest anomaly rates (~1.8%), closely followed by mobile (~1.7%). In contrast, physical card swipes at POS devices have minimal risk exposure (< 1%), indicating remote "Card-Not-Present" online environments are primary targets.
5. **High-Risk Merchant Categories (Transfers & Cash)**: Normal activity is concentrated around groceries and dining. Anomalous transactions, however, heavily target bank transfers and cash withdrawals, representing high-velocity cash-out channels after accounts are compromised.
6. **Clean Pipeline Health**: The data quality assessment confirmed that there are zero missing records and zero duplicate rows. This guarantees that any anomaly detection flags are based on true signal deviations rather than data corruption or transport issues.

## 6. Preprocessing Pipeline Step-by-Step Demonstration

Below we demonstrate each step of our modular preprocessing pipeline implemented in `src/data_preprocessing.py`.

In [ ]:
from src.data_preprocessing import (
    load_raw_data,
    clean_duplicates,
    handle_missing_values,
    validate_and_convert_types,
    standardize_categorical_values,
    detect_invalid_values
)

# Step 1: Load raw data
df_raw = load_raw_data(config.RAW_DATA_PATH)
print("Raw shape:", df_raw.shape)

### Step 2: Clean Duplicates
Detects and removes duplicate rows from the dataset.

In [ ]:
df_cleaned_dup = clean_duplicates(df_raw)

### Step 3: Handle Missing Values
Handles missing values by filling numerical features with their median and categorical features with their mode.

In [ ]:
df_imputed = handle_missing_values(df_cleaned_dup)

### Step 4: Validate and Convert Types
Ensures column datatypes are correct (e.g., converting strings to datetime for timestamps).

In [ ]:
df_typed = validate_and_convert_types(df_imputed)
print(df_typed.dtypes)

### Step 5: Standardize Categorical Values
Standardizes string representations by lowercasing and trimming whitespaces.

In [ ]:
df_std = standardize_categorical_values(df_typed)
df_std[['merchant_category', 'location', 'device_type']].head()

### Step 6: Detect Invalid/Unrealistic Values
Identifies and handles unrealistic values, such as negative or zero transaction amounts.

In [ ]:
df_final = detect_invalid_values(df_std)
print("Final shape after preprocessing:", df_final.shape)

## 7. Feature Engineering & ML Preprocessing Pipeline

In this section, we apply feature engineering and build a complete machine learning preprocessing pipeline using scikit-learn.

### 7.1. Feature Engineering
We implement the feature engineering strategy to add predictive signal:
1. **Temporal Features**: `hour_of_day`, `day_of_week`, and `is_weekend` to catch anomalies at unusual times.
2. **Geographical Features**: `is_international` (1 if location does not end in 'us', else 0) to flag higher risk cross-border activity.
3. **Customer Velocity & Spending History**: 
   - `customer_txn_count_30d`: Count of transactions by the customer over a 30-day rolling lookback window.
   - `customer_avg_amount_30d`: Average transaction amount by the customer over a 30-day rolling lookback window.
   - `amount_ratio_to_avg`: The ratio of the current transaction amount to the customer's 30-day average.

In [ ]:
from src.feature_engineering import engineer_features

df_feat = engineer_features(df_final)
df_feat[['customer_id', 'timestamp', 'amount', 'customer_txn_count_30d', 'customer_avg_amount_30d', 'amount_ratio_to_avg', 'is_international']].head(10)

### 7.2. Conceptual Explanations

#### 1. Why Feature Scaling is Needed
Features in this dataset have vastly different ranges. For example, `customer_txn_count_30d` spans small integer values (1 to 20+), while `amount` can reach up to $20,000. Many machine learning algorithms (e.g., Logistic Regression, Support Vector Machines, Neural Networks, K-Nearest Neighbors) are sensitive to feature scales. If features are not scaled, variables with larger magnitudes will dominate the distance metrics and parameter updates, preventing the model from learning from other highly predictive but smaller-scaled features. We use `StandardScaler` to transform numerical features so they have a mean of 0 and standard deviation of 1.

#### 2. Why Categorical Encoding is Needed
Machine learning algorithms operate on mathematical matrices and require numerical inputs. Categorical columns (like `merchant_category`, `device_type`, `location`) are text strings and cannot be directly fed into models. We use `OneHotEncoder` to transform each categorical variable into multiple binary indicators (dummy variables), allowing the models to interpret differences in categories without assuming an arbitrary numerical order (which would happen with label encoding).

#### 3. Why Stratified Splitting is Used
Our target variable `is_anomaly` is extremely imbalanced (only 1.50% anomalies in the raw data). In standard random splitting, there is a risk that the training set or testing set receives a disproportionate number of anomalies, or none at all. This would result in poor model training or highly volatile evaluation metrics. Stratified splitting (`stratify=y`) ensures that both the training and testing sets maintain the exact same proportion of normal-to-anomaly classes (~1.86% in cleaned datasets), providing robust and reliable model evaluation.

### 7.3. Fitting the ML Prep Pipeline & Saving Splits
We execute the preprocessing and splitting pipeline. This saves `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `split_dimensions.json`, and the preprocessor object `preprocessor.joblib`.

In [ ]:
from src.data_preprocessing import prepare_ml_dataset
import json

# This runs the splitting, ColumnTransformer preprocessing (StandardScaler + OneHotEncoder), and saves everything
prepare_ml_dataset(df_feat)

# Load and print split dimensions metadata
with open('../data/processed/split_dimensions.json', 'r') as f:
    dims = json.load(f)
print("\n=== Saved Train/Test Dataset Details ===")
print(json.dumps(dims, indent=4))

## 8. Baseline Logistic Regression Model

In this section, we train the first baseline machine learning model using a Logistic Regression classifier on our prepared training data and evaluate it on the test set.

In [ ]:
import joblib
import json
import pandas as pd
from IPython.display import Image, display
from src import config
from src.evaluate_model import evaluate

# Load test splits
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

# Load the trained model
model = joblib.load('../models/baseline_logistic_regression.joblib')

# Evaluate model
metrics, y_pred, y_prob = evaluate(model, X_test, y_test)

print("\n=== Displaying Saved Evaluation Plots ===")
display(Image(filename='../reports/figures/baseline_confusion_matrix.png'))
display(Image(filename='../reports/figures/baseline_roc_curve.png'))
display(Image(filename='../reports/figures/baseline_precision_recall_curve.png'))

## 9. Model Discussion & Business Interpretations

### 9.1. Strengths of Logistic Regression
- **High Interpretability**: Each feature's coefficient can be converted to an odds ratio, telling us exactly how a unit change in a feature (e.g., transaction amount or frequency) impacts the probability of a transaction being an anomaly.
- **Efficiency**: Extremely fast to train, require minimal memory footprint, and deliver microsecond inference times which is critical for real-time transaction authorization engines.
- **Strong Baseline**: Serves as a simple, standard benchmark for measuring if more complex non-linear models (e.g., Random Forests, XGBoost, or Neural Networks) justify their extra computational cost.
- **Low Risk of Overfitting**: With proper regularization (L1/L2), it behaves stably even on small datasets.

### 9.2. Limitations of Logistic Regression
- **Linear Assumption**: Assumes a linear decision boundary in log-odds space. It cannot capture non-linear relationships or complex feature interactions unless they are manually engineered.
- **Collinearity Sensitivity**: If two input features are highly correlated (e.g., `amount` and `amount_ratio_to_avg`), the model coefficients can become unstable and hard to interpret.
- **Struggles on Complex Patterns**: On highly intricate, multi-dimensional patterns (like sophisticated fraud rings that alter device type, categories, and times dynamically), a simple linear split may fail.

### 9.3. Situations Where It Performs Well
- In low-data regimes where complex models would easily overfit.
- When the features are linearly separable (e.g., if anomalies are predominantly characterized by high thresholds of amounts or international card use, which scaling and engineering make easily distinguishable).
- In regulated industries (like banking and insurance) where model decisions must be fully auditable and explainable to regulators.

### 9.4. Business Interpretation of False Positives and False Negatives
- **False Positives (FP) [Type I Error]**:
  - *Scenario*: A normal, legitimate transaction is flagged as an anomaly.
  - *Business Impact*: Customer friction. The client's transaction gets declined, their credit card may be locked, requiring them to contact customer service. This degrades user experience, increases customer support call volumes, and can cause transaction drop-offs (brand churn).
- **False Negatives (FN) [Type II Error]**:
  - *Scenario*: A real anomalous/fraudulent transaction is missed and approved as normal.
  - *Business Impact*: Direct financial loss. The bank/merchant is liable for chargebacks, lost funds, and potential security compliance penalties. It also erodes the user's trust in the platform's security.

## 10. Random Forest Baseline & Model Comparison

Here, we train a baseline Random Forest classifier using the same train/test split. Then, we perform a comparison of both baseline models.

In [ ]:
import joblib
import json
import pandas as pd
from IPython.display import Image, display
from src import config
from src.evaluate_model import evaluate

# Load test splits
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

# Load the trained Random Forest model
rf_model = joblib.load('../models/random_forest_baseline.joblib')

# Evaluate Random Forest model
metrics_rf, y_pred_rf, y_prob_rf = evaluate(rf_model, X_test, y_test)

print("\n=== Comparative Model Performance Metrics ===")
df_comp = pd.read_csv('../reports/model_comparison_metrics.csv')
display(df_comp)

print("\n=== Displaying Comparison Visualizations ===")
display(Image(filename='../reports/figures/comparison_metrics_bar_chart.png'))
display(Image(filename='../reports/figures/comparison_roc_curves.png'))

## 11. Comparative Model Discussion

### 11.1. Performance Analysis: Logistic Regression vs. Random Forest
- **Random Forest (RF) Outperforms Logistic Regression (LR)** across all metrics:
  - **Accuracy**: RF achieves 99.95% compared to LR's 99.90%.
  - **Recall**: RF achieves **100.00%** (detecting 37 out of 37 anomalies) compared to LR's **97.30%** (detecting 36 out of 37 anomalies).
  - **Precision**: RF achieves 97.37% compared to LR's 97.30%.
  - **F1-Score**: RF achieves **98.67%** compared to LR's **97.30%**.
  - **ROC-AUC**: RF reaches **100.00%** compared to LR's **99.998%**.
- For transaction risk mitigation, the **Recall score is paramount** because missing even a single fraud transaction (False Negative) results in direct financial liability. RF achieved perfect recall on this test split while generating slightly fewer False Positives (1) than LR (1).

### 11.2. Why Random Forest Outperforms Logistic Regression
- **Non-linear Decision Boundaries**: Anomalies in our dataset are simulated via combinations of thresholds (e.g. amounts between $5,000 and $20,000, or specific hours like 2:00 AM to 4:00 AM). Decision trees in a Random Forest easily represent these threshold step-functions. Logistic Regression assumes a continuous linear relationship in the log-odds, which is less optimal for step-threshold anomaly rules.
- **Feature Interaction Modeling**: Random Forest naturally models feature interactions (e.g., if a transaction is international AND the amount is large) through sequential node splits in the trees. Logistic Regression requires manual feature interaction terms to capture these joint indicators.
- **Robustness to Extreme Values**: Tree splits are rank-based rather than magnitude-based, meaning they are less influenced by extreme anomalies (e.g. $20,000 transfer spikes) during training.

### 11.3. Interpretability vs. Predictive Performance Trade-off
- **Logistic Regression**: High interpretability. The model yields clear coefficients. We can easily explain to a regulator or auditor exactly why a transaction was blocked using feature weights and odds ratios.
- **Random Forest**: Black-box nature. Although we can output feature importances, it is challenging to explain the exact decision path across 100 trees to an average customer or credit compliance officer. However, in credit risk management, the significant savings from preventing 100% of anomalies (Recall = 1.0) often outweighs the explanation friction, though regulatory requirements may dictate keeping explainable models or using explanation wrappers (like SHAP/LIME).

## 12. Systematic Hyperparameter Tuning

To optimize our Random Forest baseline, we perform a systematic grid search using `RandomizedSearchCV` with 5-fold cross-validation. We optimize for the F1-score to handle target imbalance.

In [ ]:
import joblib
import json
import pandas as pd
from IPython.display import Image, display
from src import config
from src.evaluate_model import evaluate

# Load best parameters found
with open('../reports/random_forest_best_params.json', 'r') as f:
    best_params = json.load(f)
print("=== Best Hyperparameters Found ===")
print(json.dumps(best_params, indent=4))

# Load test splits
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

# Load the tuned model
tuned_model = joblib.load('../models/random_forest_tuned.joblib')

# Evaluate tuned model
metrics_rf_tuned, y_pred_rf_tuned, y_prob_rf_tuned = evaluate(tuned_model, X_test, y_test)

print("\n=== Displaying Three-Way Comparison metrics ===")
df_comp_three = pd.read_csv('../reports/model_comparison_metrics_three_way.csv')
display(df_comp_three)

print("\n=== Displaying Comparison Visualizations ===")
display(Image(filename='../reports/figures/comparison_metrics_three_way.png'))
display(Image(filename='../reports/figures/comparison_roc_curves.png'))

## 13. Tuning & Cross-Validation Discussion

### 13.1. Why Hyperparameter Tuning Matters
- Machine learning algorithms are governed by parameters (hyperparameters) that are not learned from data, but must be configured by developers before training starts. Examples include the number of estimators (`n_estimators`) or maximum tree depth (`max_depth`) in a Random Forest.
- Proper tuning allows us to balance the **bias-variance trade-off**. If trees grow too deep (`max_depth=None` with small min-leaf constraints), they will overfit by capturing noise and outliers. Conversely, restricting trees too heavily will underfit. Systematic search (Grid or Randomized) identifies the optimal parameter combinations that maximize generalization capability.

### 13.2. Why Cross-Validation Improves Generalization
- Evaluating performance on a single validation set can be misleading due to sampling variance or lucky splits. A model might perform well on one specific split but poorly on another.
- **K-Fold Cross-Validation** (in our case, 5-fold CV) divides the training data into 5 equal portions. The model trains on 4 folds and evaluates on the remaining fold, repeating this process 5 times. Averaging these F1-scores provides a stable, low-variance estimate of performance across out-of-fold data, ensuring that the selected hyperparameters truly generalize to unseen data.

### 13.3. Signs of Overfitting vs. Underfitting
- **Overfitting (High Variance)**:
  - *Indicators*: Model achieves extremely high performance (e.g., 100% accuracy, 1.0 F1-score) on the training set, but significantly lower scores on the testing/validation set.
  - *Cause*: Model has excessive structural capacity and memorized training data, including noise, rather than learning core patterns.
- **Underfitting (High Bias)**:
  - *Indicators*: Model performs poorly on both the training and the testing splits.
  - *Cause*: Model lacks the complexity to represent the data's underlying patterns. For instance, using a linear decision boundary on highly interactive or non-linear data distributions, or restricting tree depth too severely.